google collab depedencies

In [4]:
!pip -q install bertopic
!pip -q install sastrawi
!pip -q install gensim

In [53]:
!git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
%cd topic_modeling_KBMI4

In [1]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px
import random

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

# for windows
# import umap as UMAP
# import hdbscan as HDBSCAN

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : Tesla T4


In [3]:
df = pd.read_csv("data/preprocessed_data_downsampled.csv")

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/preprocessed_data_downsampled.csv'

In [57]:
df["word_count"] = df["text"].astype(str).str.split().apply(len)
df = df[df["word_count"] >= 5].reset_index(drop=True)
print(f"Total documents setelah filter: {len(df):,}")

Total documents setelah filter: 135,913


In [58]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 135,913


# IndoBERT

In [59]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)

model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(50000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [60]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

In [61]:
def encode_documents(
    documents,
    batch_size=32,
    max_length=128
):

    embeddings = []

    with torch.no_grad():

        for i in tqdm(
            range(0, len(documents), batch_size)
        ):

            batch = documents[
                i:i+batch_size
            ]

            encoded_input = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded_input = {
                k: v.to(device)
                for k, v in encoded_input.items()
            }

            model_output = model(**encoded_input)

            sentence_embeddings = mean_pooling(
                model_output,
                encoded_input["attention_mask"]
            )

            sentence_embeddings = (
                sentence_embeddings
                .cpu()
                .numpy()
            )

            embeddings.append(sentence_embeddings)

    return np.vstack(embeddings)

In [62]:
embeddings = encode_documents(
    documents,
    batch_size=32,
    max_length=128
)

  0%|          | 0/4248 [00:00<?, ?it/s]

In [63]:
print(embeddings.shape)

(135913, 768)


In [64]:
embeddings[0]

array([ 1.99362922e+00,  8.71141791e-01,  2.32576132e-01,  4.88159537e-01,
       -1.53942794e-01,  6.54067397e-02, -7.94849396e-01,  6.44384176e-02,
        5.35462677e-01, -1.38591528e-01, -8.13294351e-02, -1.04618895e+00,
       -9.04206634e-01,  8.62347782e-01,  2.20925231e-02, -3.00665766e-01,
       -5.28008997e-01, -1.01320833e-01, -2.91478708e-02,  7.27127671e-01,
        3.72639418e-01, -3.66047174e-01, -6.55413792e-02, -1.06161618e+00,
       -7.23849773e-01, -2.64187664e-01, -7.38127232e-02,  3.38941991e-01,
       -4.32386130e-01, -4.99101728e-01,  7.13105083e-01,  4.50596094e-01,
       -7.68143870e-03,  3.48131716e-01, -1.58587146e+00,  1.17839050e+00,
       -4.46127206e-01,  1.15841889e+00, -1.05954075e+00, -1.42351031e-01,
       -1.08025956e+00,  3.64045143e-01, -1.19255567e+00, -4.37219381e-01,
       -3.45705301e-01,  7.09880710e-01,  3.18425953e-01,  1.71062803e+00,
        4.44132164e-02,  1.10976048e-01, -1.06760168e+00, -7.84462571e-01,
        2.28673473e-01,  

In [65]:
norms = np.linalg.norm(embeddings, axis=1)

print("Minimum Norm :", norms.min())
print("Maximum Norm :", norms.max())
print("Average Norm :", norms.mean())
print("Std Norm :", norms.std())

Minimum Norm : 12.87398
Maximum Norm : 26.059353
Average Norm : 17.716616
Std Norm : 1.4305303


In [66]:
print("NaN :", np.isnan(embeddings).sum())
print("Inf :", np.isinf(embeddings).sum())

NaN : 0
Inf : 0


In [67]:
np.save(
    "indobert_embeddings_downsampled.npy",
    embeddings
)

# BERTopic

In [68]:
embeddings = np.load("indobert_embeddings_downsampled.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (135913, 768)


In [69]:
sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# extra_particles = ["banget", "terus", "padahal", "sih", "aja", "saja", "dong", "deh", "ya", "kok", "biar", "gitu", "nih", "loh", "mau", "sudah", "belum"]
# sastrawi_stopwords_extended = sastrawi_stopwords + extra_particles

vectorizer_model = CountVectorizer(
  ngram_range=(1,2),
  stop_words=sastrawi_stopwords,
  token_pattern=r"(?u)\b[^\d\W]+\b",
  min_df=2,
  )

baseline UMAP for testing purpose

In [70]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [71]:
hdbscan_model = HDBSCAN(
    min_cluster_size=100,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="leaf",
    prediction_data=True
)

In [72]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

In [73]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-10 07:18:02,112 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-10 07:18:12,333 - BERTopic - Dimensionality - Completed ✓
2026-08-10 07:18:12,337 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-10 07:18:14,989 - BERTopic - Cluster - Completed ✓
2026-08-10 07:18:15,020 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-10 07:18:19,307 - BERTopic - Representation - Completed ✓


In [74]:
new_topics = topic_model.reduce_outliers(
    documents, topics,
    strategy="embeddings",
    embeddings=embeddings,   # <- pakai embedding IndoBERT asli
)
topic_model.update_topics(documents, topics=new_topics)
topics = new_topics

2026-08-10 07:18:21,985 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outlier setelah reduce (embeddings strategy): 0 (0.00%)


In [ ]:
original_outliers = np.sum(original_topics == -1)
new_outliers = np.sum(topics == -1)

original_pct = original_outliers / len(original_topics) * 100
new_pct = new_outliers / len(topics) * 100

print("=" * 70)
print("OUTLIER REDUCTION")
print("=" * 70)

print(
    f"Before : {original_outliers:,} "
    f"({original_pct:.2f}%)"
)

print(
    f"After  : {new_outliers:,} "
    f"({new_pct:.2f}%)"
)

print(
    f"Recovered : "
    f"{original_outliers - new_outliers:,} reviews"
)

# Evaluation

Basic Statistics

In [75]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,0,4278,0_wajah_verifikasi_gagal_susah,"[wajah, verifikasi, gagal, susah, terus, foto, selalu, ktp, verivikasi, muka]","[verifikasi wajah kenapa gagal terus, verifikasi wajah gagal terus kenapa ini, kenapa verifikasi wajah tidak bisa]"
1,1,5043,1_potongan_biaya_saldo_admin,"[potongan, biaya, saldo, admin, ribu, bulan, ada, potong, uang, 50]","[banyak potongan potongan bulanan mahal apalagi tengah bulan potongan lagi debit, banyak potongan ya 1bulan biasa potongan 17500 lah menambah lagi potongan 11500 biaya kartu jadi total 29 ribu makanya saldo jarang saya isi tolong di perbaiki sebelum tidak ada biaya seperti itu kalo potongan tiap bulan wajar, bank yang terlalu banyak potongan dari biaya admin sudah di potong 10 tiap bulan dan di potong lagi enggak jelasnya di potongnya juga lumayan bagi saya terlalu banya dengn saldo yang cuma 1 digit potongan biaya kartu debit 18 ribu tiap bulan taila]"
2,2,3727,2_tolong_perbaiki_diperbaiki_mohon,"[tolong, perbaiki, diperbaiki, mohon, di, segera, dong, lagi, update, sering]","[setelah di update aplikasi selalu force close tolong segera di perbaiki, mohon segera di perbaiki tidak bisa login, force close mohon segera diperbaiki]"
3,3,3563,3_password_username_benar_salah,"[password, username, benar, salah, pasword, login, sandi, padahal, sudah, lupa]","[kenapa tidak bisa login padahal username sama password sudah benar tapi tidak bisa masuk username dan password salah, kenapa tidak bisa login padahal username dan password sudah benar, kenapa saya tidak bisa login padahal username dan password benar]"
4,4,3047,4_update_meminta_sering_terlalu,"[update, meminta, sering, terlalu, tiap, terus, bulan, dikit, lemot, harus]","[setiap bulan meminta update terus, terlalu sering update update update, meminta di update terus _]"
5,5,3722,5_kenapa_buka_bisa_ya,"[kenapa, buka, bisa, ya, dibuka, kok, aplikasi, di, update, enggak]","[kenapa kok enggak bisa buka aplikasi ini, kok enggak bisa di buka ya aplikasi nya, kok enggak bisa di buka aplikasi nya]"
6,6,9103,6_update_aplikasi_tetap_bisa,"[update, aplikasi, tetap, bisa, sudah, di, buka, tidak, setelah, lagi]","[force close terus sudah clear cache sudah clear data sudah uninstall dan install kembali masih force close terus, kok force close terus bagaimana nih saya mau tf tidak bisa mau login force close lagi sampai saya hapus data uninstall habis itu instal lagi masih tetap saja force close, setelah update aplikasi nya setiap dibuka pasti langsung keluar sendiri saya enggak bisa buka aplikasi bca mobile sudah saya uninstall dan install ulang masih tetap sama saja]"
7,7,3026,7_susah_daftar_mau_saja,"[susah, daftar, mau, saja, ribet, banget, bikin, ampun, gagal, buat]","[ribet banget mau daftar juga susah, mau daftar saja susah harus ulang2, mau daftar saja susah banget]"
8,8,3835,8_mulu_banget_eror_lemot,"[mulu, banget, eror, lemot, jelas, parah, jelek, gangguan, sering, aplikasi]","[sering banget aplikasi eror enggak jelas, aplikasi enggak jelas eror mulu, eror mulu enggak jelas juga]"
9,9,1083,9_otp_kode_kirim_dikirim,"[otp, kode, kirim, dikirim, sms, masuk, email, menerima, meminta, enggak]","[kode otp enggak masuk masuk kenapa ya, kode otp nya enggak sampai sampai, kode otp enggak ke kirim2]"


In [76]:
num_topics = len(topic_info) - 1

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 61
Outliers            : 0
Outlier Percentage  : 0.00%


Topic Size

In [77]:
topic_info[["Topic","Count"]]

,Topic,Count
0,0,4278
1,1,5043
2,2,3727
3,3,3563
4,4,3047
...,...,...
57,57,561
58,58,760
59,59,475
60,60,489


Top Words

In [78]:
for topic in topic_info.Topic:

    if topic == -1:
        continue

    print("="*80)

    print(f"Topic {topic}")

    print(topic_model.get_topic(topic))

Topic 0
[('wajah', np.float64(0.10984204272392974)), ('verifikasi', np.float64(0.08344762824753779)), ('gagal', np.float64(0.06049389763805774)), ('susah', np.float64(0.032095289445985886)), ('terus', np.float64(0.027694180048339425)), ('foto', np.float64(0.025956220031620528)), ('selalu', np.float64(0.02393834217308296)), ('ktp', np.float64(0.021931802585110387)), ('verivikasi', np.float64(0.021573003596601106)), ('muka', np.float64(0.020102769471372947))]
Topic 1
[('potongan', np.float64(0.034061037567090134)), ('biaya', np.float64(0.03345842703923455)), ('saldo', np.float64(0.02708538918218008)), ('admin', np.float64(0.022113321419925478)), ('ribu', np.float64(0.021705274030583338)), ('bulan', np.float64(0.0190568056844629)), ('ada', np.float64(0.016388620254306926)), ('potong', np.float64(0.016253115935565294)), ('uang', np.float64(0.013927717667540944)), ('50', np.float64(0.01272080129656058))]
Topic 2
[('tolong', np.float64(0.0635150094855986)), ('perbaiki', np.float64(0.05868403

Representative Reviews

In [79]:
representative_docs = topic_model.get_representative_docs()

for topic in representative_docs:

    if topic == -1:
        continue

    print("="*100)

    print(f"Topic {topic}")

    print()

    for doc in representative_docs[topic][:5]:

        print("-", doc)

    print()

Topic 0

- verifikasi wajah kenapa gagal terus
- verifikasi wajah gagal terus kenapa ini
- kenapa verifikasi wajah tidak bisa

Topic 1

- banyak potongan potongan bulanan mahal apalagi tengah bulan potongan lagi debit
- banyak potongan ya 1bulan biasa potongan 17500 lah menambah lagi potongan 11500 biaya kartu jadi total 29 ribu makanya saldo jarang saya isi tolong di perbaiki sebelum tidak ada biaya seperti itu kalo potongan tiap bulan wajar
- bank yang terlalu banyak potongan dari biaya admin sudah di potong 10 tiap bulan dan di potong lagi enggak jelasnya di potongnya juga lumayan bagi saya terlalu banya dengn saldo yang cuma 1 digit potongan biaya kartu debit 18 ribu tiap bulan taila

Topic 2

- setelah di update aplikasi selalu force close tolong segera di perbaiki
- mohon segera di perbaiki tidak bisa login
- force close mohon segera diperbaiki

Topic 3

- kenapa tidak bisa login padahal username sama password sudah benar tapi tidak bisa masuk username dan password salah
- kenapa

silhoutte score

In [80]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : -0.0001


In [81]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info.Topic:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]
    topic_words.append(words)

unique_words = len(
    set(chain.from_iterable(topic_words))
)

total_words = len(topic_words) * top_n
topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.4548


Representative Reviews

In [82]:
representative_docs = topic_model.get_representative_docs()

for topic, docs in representative_docs.items():

    if topic == -1:
        continue

    print("="*100)

    print(f"TOPIC {topic}")

    print()

    for i, doc in enumerate(docs[:5],1):

        print(f"{i}. {doc}")

        print()

TOPIC 0

1. verifikasi wajah kenapa gagal terus

2. verifikasi wajah gagal terus kenapa ini

3. kenapa verifikasi wajah tidak bisa

TOPIC 1

1. banyak potongan potongan bulanan mahal apalagi tengah bulan potongan lagi debit

2. banyak potongan ya 1bulan biasa potongan 17500 lah menambah lagi potongan 11500 biaya kartu jadi total 29 ribu makanya saldo jarang saya isi tolong di perbaiki sebelum tidak ada biaya seperti itu kalo potongan tiap bulan wajar

3. bank yang terlalu banyak potongan dari biaya admin sudah di potong 10 tiap bulan dan di potong lagi enggak jelasnya di potongnya juga lumayan bagi saya terlalu banya dengn saldo yang cuma 1 digit potongan biaya kartu debit 18 ribu tiap bulan taila

TOPIC 2

1. setelah di update aplikasi selalu force close tolong segera di perbaiki

2. mohon segera di perbaiki tidak bisa login

3. force close mohon segera diperbaiki

TOPIC 3

1. kenapa tidak bisa login padahal username sama password sudah benar tapi tidak bisa masuk username dan passwor

In [83]:
random.seed(42)

sample_size = 20

for topic_id in sorted(set(topics)):

    if topic_id == -1:
        continue

    topic_docs = [
        doc for doc, topic in zip(documents, topics)
        if topic == topic_id
    ]

    n = min(sample_size, len(topic_docs))
    sampled_docs = random.sample(topic_docs, n)

    print("\n" + "=" * 120)
    print(f"TOPIC {topic_id}")
    print(f"CLUSTER SIZE : {len(topic_docs)}")
    print(f"SAMPLE SIZE  : {n}")
    print("=" * 120)

    for i, doc in enumerate(sampled_docs, 1):
        print(f"{i}. {doc}")


TOPIC 0
CLUSTER SIZE : 4278
SAMPLE SIZE  : 20
1. kenapa kok registrasi brimo terlalu banyak syaratnya foto sana foto sini bikin ribet
2. enggak bisa scan wajah ini bagaimana
3. verifikasi wajah gagal terus padahal pencahayaan bagus
4. kenapa verifikasi wajah susah benar di hp sinyal penuh stabil selalu stuck saja
5. susah verifikasi wajah sampai stres enggak bisa bisa
6. sangat buruk tidak bisa verifikasi wajah ber ulang ulang kali
7. verifikasi wajah najis 5x kagak bsa2
8. verifikasi wajah selalu gagal padahal sudah di bawah matahari
9. mulai memburuk semua proses gagal
10. verifikasi wajah gagal mulu padahal muka juga enggak operasi plastik atau sejenisnya tapi susah banget buat verfikasj
11. ini kenapa setiap verifikas wajah selalu gagal terus tidak bisa buat verifikasi
12. aplikasi lemot saat verifikasi wajah bolak balik terus enggak bisa
13. verifikasi wajah sangat sulit dilakukan
14. aplikasi paling susah data sudah betul masih saja gagal
15. verifikasi wajah gagal mulu enggak n

NPMI

In [84]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [85]:
doc.split()

['enggak', 'bisa', 'dipakai', 'transaksi', 'pemblokiran', 'sepihak']

In [86]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [87]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)
topic_words = []

for topic in topic_info.Topic:

    if topic == -1:
        continue

    words = []

    for word, score in topic_model.get_topic(topic):
        if word in dictionary.token2id:
            words.append(word)
    # Need at least 2 words for coherence
    if len(words) >= 2:
        topic_words.append(words)

In [88]:
# sanity check
print(f"Valid Topics : {len(topic_words)}")

print()

print(topic_words[:3])

Valid Topics : 62

[['wajah', 'verifikasi', 'gagal', 'susah', 'terus', 'foto', 'selalu', 'ktp', 'verivikasi', 'muka'], ['potongan', 'biaya', 'saldo', 'admin', 'ribu', 'bulan', 'ada', 'potong', 'uang', '50'], ['tolong', 'perbaiki', 'diperbaiki', 'mohon', 'di', 'segera', 'dong', 'lagi', 'update', 'sering']]


In [89]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : 0.1010


DBCV

In [90]:
mask = np.array(topics) != -1
X = topic_model.umap_model.embedding_[mask].astype(np.float64)
labels = np.array(topics)[mask]

dbcv_score = validity_index(X, labels)
print(f"DBCV : {dbcv_score:.4f}")

DBCV : -0.9244


In [91]:
import pandas as pd
from scipy.stats import chi2_contingency

df["topic"] = topics

# 1. Baseline: proporsi tiap bank di keseluruhan korpus
baseline = df["bank"].value_counts(normalize=True) * 100
print("Proporsi bank di keseluruhan korpus (baseline):")
print(baseline.round(2))
print()

# 2. Proporsi tiap bank DI DALAM tiap topik
crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
crosstab = crosstab.round(2)

# 3. Hitung "lift" = proporsi di topik / proporsi baseline
#    >1 artinya over-represented di topik itu, <1 artinya under-represented
lift = crosstab.copy()
for bank in baseline.index:
    lift[bank] = crosstab[bank] / baseline[bank]

# 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
def flag_imbalance(row):
    return any(row > 1.5) or any(row < 0.5)

lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
lift["topic_size"] = lift.index.map(topic_sizes)

result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])

Proporsi bank di keseluruhan korpus (baseline):
bank
BRIMO_REVIEWS            31.00
LIVIN_MANDIRI_REVIEWS    30.92
BCAMOBILE_REVIEWS        24.81
WONDR_BNI_REVIEWS        13.27
Name: proportion, dtype: float64

bank   BRIMO_REVIEWS  LIVIN_MANDIRI_REVIEWS  BCAMOBILE_REVIEWS  \
topic                                                            
0           1.131802               0.538167           1.137076   
51          1.649160               0.509706           0.518355   
33          1.407576               0.585709           0.523998   
36          0.520584               1.046579           0.576398   
37          2.732581               0.182731           0.334956   
...              ...                    ...                ...   
17          0.737977               1.179827           1.199956   
16          1.079228               0.907833           1.029052   
39          1.408221               1.105765           0.582444   
15          0.754750               1.335391           1.033082 

looking for outlier

In [92]:
topics_array = np.array(topics)
outlier_mask = topics_array == -1

outlier_df = pd.DataFrame({
    "original_index": np.where(outlier_mask)[0],
    "text": np.array(documents)[outlier_mask]
})

print(f"Total outliers: {len(outlier_df):,}")

outlier_df.head(20)

Total outliers: 0


,original_index,text


In [93]:
random_outliers = outlier_df.sample(
    n=min(100, len(outlier_df)),
    random_state=42
).reset_index(drop=True)

pd.set_option("display.max_colwidth", None)

random_outliers

,original_index,text


In [94]:
outlier_df["word_count"] = (
    outlier_df["text"]
    .str.split()
    .str.len()
)

outlier_df["char_count"] = (
    outlier_df["text"]
    .str.len()
)

outlier_df[["word_count", "char_count"]].describe()

,word_count,char_count
count,0.0,0.0
mean,NaN,NaN
std,NaN,NaN
min,NaN,NaN
25%,NaN,NaN
50%,NaN,NaN
75%,NaN,NaN
max,NaN,NaN


In [95]:
outlier_df.sort_values(
    "word_count",
    ascending=True
).head(50)[
    ["original_index", "word_count", "text"]
]

,original_index,word_count,text


In [96]:
outlier_df.sort_values(
    "word_count",
    ascending=False
).head(50)[
    ["original_index", "word_count", "text"]
]

,original_index,word_count,text
